## MININET_TEST

In [2]:
import torch
from torch import nn

In [3]:
x = torch.randn(100, 3)
layer = nn.Linear(3, 5)
print(layer(x).shape)
print(layer.weight)
print(layer.bias)

torch.Size([100, 5])
Parameter containing:
tensor([[ 0.5702, -0.1754, -0.1922],
        [ 0.4425,  0.0187, -0.0541],
        [ 0.0628, -0.4785,  0.1006],
        [-0.1086, -0.1965,  0.2813],
        [-0.5360,  0.0853, -0.2138]], requires_grad=True)
Parameter containing:
tensor([-0.4219, -0.3476,  0.5662,  0.3059, -0.1469], requires_grad=True)


In [ ]:
# relu를 통과하면 음수인 애들이 진짜 사라질까?
x = torch.randn(2, 5)
print(x)
layer = nn.ReLU()
print(layer(x)) # 사라진다!!

tensor([[ 0.3710, -0.8065, -1.1122, -0.1563,  0.5394],
        [ 0.3563, -0.3886, -1.9934,  1.9965, -0.1347]])
tensor([[0.3710, 0.0000, 0.0000, 0.0000, 0.5394],
        [0.3563, 0.0000, 0.0000, 1.9965, 0.0000]])


In [ ]:
x = torch.randn(3, 7)
print(x)
drop = nn.Dropout(p = 0.2) # 논문은 살릴 확률 --> 구현은 죽일 확률로 된다. train과 test시에 다르게 동작한다.
print(drop(x))

tensor([[ 0.2129, -0.1067, -0.6676, -0.5193,  1.7760, -0.8224, -0.9544],
        [ 1.4141, -0.4204,  0.6228, -0.3818,  0.8972, -0.4367, -0.0284],
        [ 0.5522, -0.1712, -0.1737,  0.9328,  1.7288,  0.0262, -0.3631]])
tensor([[ 0.2661, -0.1334, -0.0000, -0.6492,  2.2199, -1.0280, -1.1930],
        [ 0.0000, -0.5255,  0.7785, -0.4773,  0.0000, -0.5458, -0.0355],
        [ 0.6902, -0.0000, -0.2171,  0.0000,  0.0000,  0.0000, -0.4538]])


### Dropout의 두 가지 동작
Dropout은 단순히 값을 0으로 만드는 것만 하지 않아요.
1. 일부 뉴런을 0으로 만든다 (p=0.2 → 20% 확률로 0)
2. 살아남은 값들을 1/(1-p) 배로 스케일 업한다
p=0.2이면 → 1/(1-0.2) = 1/0.8 = 1.25배 곱해줘요.

왜 스케일 업을 할까?
예를 들어 값이 10개 있고 p=0.2라면:

Dropout 적용 시 → 2개가 0이 되고, 8개만 살아남음
근데 그냥 두면 전체 합이 줄어들어버림 (원래 합의 80%밖에 안 됨)
그러면 학습할 때와 추론할 때 값의 스케일이 달라지는 문제 발생

그래서 살아남은 값에 1.25를 곱해서 "기댓값을 원래랑 똑같이" 유지시켜줘요. 

E[dropout(x)] = (1-p) * x * 1/(1-p) = x

기댓값이 x로 보존됨! 

In [6]:
class sample_model(nn.Module):
    def __init__(self):
        super().__init__()
        self.drop_layer = nn.Sequential(nn.Linear(5, 7),
                                        nn.Dropout(p=0.3))
    def forward(self, x):
        x = self.drop_layer(x)
        return x
    
model = sample_model()
model.train() # train mode
x = torch.randn(3, 5)
print(model(x))

model.eval() # test mode
print(model(x))

tensor([[-0.1046,  0.2901,  0.5550,  0.0000, -0.0000, -0.4383,  0.2932],
        [ 0.3994, -0.0058,  0.5518,  1.5063, -1.6802, -0.9664, -0.0259],
        [-0.8771,  0.6148,  0.0000, -0.7782,  0.1385,  0.5573,  1.4613]],
       grad_fn=<MulBackward0>)
tensor([[-0.0732,  0.2031,  0.3885,  0.5633, -0.3563, -0.3068,  0.2052],
        [ 0.2796, -0.0041,  0.3863,  1.0544, -1.1761, -0.6765, -0.0182],
        [-0.6140,  0.4303,  0.4315, -0.5448,  0.0969,  0.3901,  1.0229]],
       grad_fn=<AddmmBackward0>)


In [8]:
layer = nn.Conv2d(in_channels=1, out_channels=2, kernel_size=3, stride=1, padding=1) # stride=1, padding=0 이 디폴트
layer(torch.randn(32, 1, 5, 5)).shape  # 5x5 흑백 사진 32장 -> 개 채 행 열
# nn.Linear(3,5) # 채채 # 얘는 채 또는 개채를 원함, 개 x 3
# nn.Conv2d(3,5) # 채채 # 얘는 채행열 또는 개채행열을 원함, 개 x 3 x 행 x 열

torch.Size([32, 2, 5, 5])

In [10]:
layer = nn.Conv2d(3, 5, 3, stride = 2, padding = 1)
x = torch.randn(32, 3, 5, 5)
print(layer(x).shape)

torch.Size([32, 5, 3, 3])


In [ ]:
conv1 = nn.Conv2d(1,8,6,stride=2)
x=torch.randn(32,1,28,28)
print(conv1(x).shape)

conv2 = nn.Conv2d(8,16,3,padding=1)
print(conv2(conv1(x)).shape)

Maxpool = nn.MaxPool2d(kernel_size=2, stride=(2,2)) # Kernel_size -> pooling할 전체에서의 subset size
print(Maxpool(conv2(conv1(x))).shape)

In [ ]:
maxpool = nn.MaxPool2d(2) # 2로만 줘도 자동 Kernel_size = 2, stride = (2,2), stride --> kernel size랑 동일하게 됨
x = torch.randn(1, 1, 8, 8)
print(x)
print(f"The shape of x {x.shape}")

print(f"Print x after Maxpooling {maxpool(x)}")
print(f"The size of x after maxpooling {maxpool(x).shape}")

tensor([[[[-0.3199, -0.7104,  0.1279,  0.2563, -0.7370, -0.8617, -0.1205,
            1.0294],
          [-1.4976,  0.9563, -0.8779,  0.9538,  0.4869,  0.2580, -0.7144,
            0.7995],
          [ 0.1339, -1.2413, -0.7944,  0.6378,  1.2917,  0.8613, -1.6189,
           -2.2348],
          [ 1.5719,  1.4267, -0.0842, -1.0729,  0.3327, -0.6573,  0.0473,
            0.2653],
          [-0.6084,  0.8086, -0.3914,  0.1670,  0.3385, -0.5156, -0.8732,
           -0.2007],
          [-0.2743,  0.7891,  0.5615,  0.8585,  0.3187, -0.2526, -0.7648,
           -1.1794],
          [-0.4578, -0.4584, -0.5877, -0.8431, -0.0390, -1.2049, -0.5138,
           -0.7831],
          [-0.4716,  0.0722, -0.9267, -0.5166,  0.4752, -0.8452, -0.2886,
            0.9794]]]])
The shape of x torch.Size([1, 1, 8, 8])
Print x after Maxpooling tensor([[[[1.5719, 1.2917],
          [0.8585, 0.9794]]]])
The size of x after maxpooling torch.Size([1, 1, 2, 2])


In [15]:
avgpool = nn.AvgPool2d(2)
x = torch.randn(1, 1, 6, 6)
print(x)
print(avgpool(x))
print(avgpool(torch.randn(32, 3, 6, 6)).shape)


tensor([[[[ 2.0441,  0.7915,  1.0976, -1.1787,  0.3083, -0.3585],
          [ 0.4775, -0.7079, -0.2382, -2.0166, -0.3430,  1.8383],
          [ 0.2555, -0.2983, -0.8738,  0.5000, -0.7790, -0.2456],
          [-0.1666, -0.2336,  1.6232,  1.1316, -1.1740, -2.4889],
          [ 0.3950,  0.7209,  0.0664, -1.7758, -0.8074,  0.5040],
          [ 0.9829, -0.1956,  0.1007,  0.9226, -0.5522, -0.0140]]]])
tensor([[[[ 0.6513, -0.5840,  0.3613],
          [-0.1107,  0.5952, -1.1719],
          [ 0.4758, -0.1715, -0.2174]]]])
torch.Size([32, 3, 3, 3])


In [26]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.conv1 = nn.Conv2d(1, 8, 6, stride = 2)
        self.conv2 = nn.Conv2d(8, 16, 3, padding =1)
        self.Maxpool2 = nn.MaxPool2d(2)
        self.fc = nn.Linear(16 * 6 * 6,10)
        
    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.Maxpool2(x)
        x = torch.flatten(x, start_dim=1)
        x = self.fc(x)
        return x

x = torch.randn(32, 1, 28, 28)
model = CNN()
print(model(x).shape)
        
        

torch.Size([32, 10])


In [25]:
x = torch.randn(32, 2, 8, 8)
print(torch.flatten(x, start_dim=0).shape) # 0번째 차원부터 끝까지 전부 하나로 펼친다..
print(torch.flatten(x, start_dim=1).shape) # 1번째 차원부터 끝까지 전부 하나로 펼친다..


torch.Size([4096])
torch.Size([32, 128])


## .parameters() vs .modules() vs .children() 그리고 isinstance의 활용

In [2]:
import torch
from torch import nn

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Sequential(nn.Linear(2, 3),
                                 nn.ReLU())
        self.fc2 = nn.Sequential(nn.Linear(3, 4),
                                 nn.ReLU())
        self.fc_out = nn.Sequential(nn.Linear(4, 1),
                                    nn.Sigmoid())
        
    def forward(self, x):
        x = self.fc1(x)
        x = self.fc2(x)
        x = self.fc_out(x)
        return x

model = MLP()
print(model(torch.randn(2, 2)).shape)
print(model)

torch.Size([2, 1])
MLP(
  (fc1): Sequential(
    (0): Linear(in_features=2, out_features=3, bias=True)
    (1): ReLU()
  )
  (fc2): Sequential(
    (0): Linear(in_features=3, out_features=4, bias=True)
    (1): ReLU()
  )
  (fc_out): Sequential(
    (0): Linear(in_features=4, out_features=1, bias=True)
    (1): Sigmoid()
  )
)


In [ ]:
list(model.parameters())[0]
# [layer0 weight 값, layer0 bias 값, layer1 weight 값, layer1 bias 값, ...]

Parameter containing:
tensor([[-0.1706,  0.2726],
        [ 0.6198, -0.7060],
        [ 0.1978,  0.1526]], requires_grad=True)

In [9]:
# for tranfer learning
model = MLP()
print([p for p in model.parameters() if p.requires_grad])


[Parameter containing:
tensor([[ 0.5838, -0.3974],
        [-0.3794, -0.3880],
        [-0.6226, -0.0601]], requires_grad=True), Parameter containing:
tensor([-0.2502,  0.0960, -0.2517], requires_grad=True), Parameter containing:
tensor([[-0.0314, -0.5483, -0.2000],
        [ 0.1941, -0.0807,  0.2745],
        [-0.4218,  0.1392,  0.1371],
        [-0.1302, -0.0799,  0.3652]], requires_grad=True), Parameter containing:
tensor([-0.3476,  0.3772, -0.3896,  0.4676], requires_grad=True), Parameter containing:
tensor([[-0.2397, -0.2827, -0.3730, -0.1699]], requires_grad=True), Parameter containing:
tensor([0.1241], requires_grad=True)]


In [10]:
# 전체 freeze
for p in model.parameters():
    p.requires_grad = False
print([p for p in model.parameters() if p.requires_grad])

[]


In [13]:
# 일부만 freeze(즉 일부만 수정할때)
for p in model.parameters():
    p.requires_grad = False
# fc_out 레이어만 새로 교체
model.fc_out = nn.Linear(4, 10)
params = [p for p in model.parameters() if p.requires_grad]
print(params)

# 이놈들만 훈련
from torch import optim
optimizer = optim.Adam(params, lr= 0.1)

[Parameter containing:
tensor([[ 0.2608, -0.0988, -0.1019, -0.1606],
        [-0.2428, -0.4241,  0.1449,  0.1081],
        [-0.0759, -0.4647, -0.3640, -0.2663],
        [-0.2366,  0.1064, -0.4806, -0.1754],
        [-0.2484, -0.4129,  0.3865,  0.1109],
        [ 0.2359,  0.0825, -0.0922,  0.2564],
        [-0.3772,  0.1182,  0.1334, -0.3432],
        [-0.4387,  0.0751,  0.2360,  0.3409],
        [ 0.2963,  0.0590,  0.1928, -0.3973],
        [ 0.3484, -0.2057,  0.1416, -0.3823]], requires_grad=True), Parameter containing:
tensor([ 0.4572,  0.4147, -0.2881,  0.4387,  0.4760, -0.1187,  0.2730,  0.0528,
        -0.0149,  0.4461], requires_grad=True)]


In [14]:
list(model.modules())

[MLP(
   (fc1): Sequential(
     (0): Linear(in_features=2, out_features=3, bias=True)
     (1): ReLU()
   )
   (fc2): Sequential(
     (0): Linear(in_features=3, out_features=4, bias=True)
     (1): ReLU()
   )
   (fc_out): Linear(in_features=4, out_features=10, bias=True)
 ),
 Sequential(
   (0): Linear(in_features=2, out_features=3, bias=True)
   (1): ReLU()
 ),
 Linear(in_features=2, out_features=3, bias=True),
 ReLU(),
 Sequential(
   (0): Linear(in_features=3, out_features=4, bias=True)
   (1): ReLU()
 ),
 Linear(in_features=3, out_features=4, bias=True),
 ReLU(),
 Linear(in_features=4, out_features=10, bias=True)]

In [15]:
print([m for m in model.modules() if isinstance(m,nn.Linear)])
print([m.weight for m in model.modules() if isinstance(m,nn.Linear)])
print([m.weight.grad for m in model.modules() if isinstance(m,nn.Linear)])

[Linear(in_features=2, out_features=3, bias=True), Linear(in_features=3, out_features=4, bias=True), Linear(in_features=4, out_features=10, bias=True)]
[Parameter containing:
tensor([[ 0.5838, -0.3974],
        [-0.3794, -0.3880],
        [-0.6226, -0.0601]]), Parameter containing:
tensor([[-0.0314, -0.5483, -0.2000],
        [ 0.1941, -0.0807,  0.2745],
        [-0.4218,  0.1392,  0.1371],
        [-0.1302, -0.0799,  0.3652]]), Parameter containing:
tensor([[ 0.2608, -0.0988, -0.1019, -0.1606],
        [-0.2428, -0.4241,  0.1449,  0.1081],
        [-0.0759, -0.4647, -0.3640, -0.2663],
        [-0.2366,  0.1064, -0.4806, -0.1754],
        [-0.2484, -0.4129,  0.3865,  0.1109],
        [ 0.2359,  0.0825, -0.0922,  0.2564],
        [-0.3772,  0.1182,  0.1334, -0.3432],
        [-0.4387,  0.0751,  0.2360,  0.3409],
        [ 0.2963,  0.0590,  0.1928, -0.3973],
        [ 0.3484, -0.2057,  0.1416, -0.3823]], requires_grad=True)]
[None, None, None]


In [18]:
# weight initialization에 활용
for m in model.modules():
    if isinstance(m, nn.Linear):
        #nn.init.kaiming_normal_(m.weight)
        nn.init.constant_(m.weight, 1)

print([m.weight for m in model.modules() if isinstance(m, nn.Linear)])

[Parameter containing:
tensor([[1., 1.],
        [1., 1.],
        [1., 1.]]), Parameter containing:
tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]]), Parameter containing:
tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]], requires_grad=True)]


In [19]:
list(model.children())

[Sequential(
   (0): Linear(in_features=2, out_features=3, bias=True)
   (1): ReLU()
 ),
 Sequential(
   (0): Linear(in_features=3, out_features=4, bias=True)
   (1): ReLU()
 ),
 Linear(in_features=4, out_features=10, bias=True)]

In [20]:
x = torch.randn(2,2)
list(model.children())[0](x)

tensor([[0.0000, 0.1177, 0.0000],
        [0.7707, 1.1169, 0.7693]])

In [21]:
print(*list(model.children())[:2])
sub_network = nn.Sequential(*list(model.children())[:2])
print(sub_network)
print(sub_network(x))

Sequential(
  (0): Linear(in_features=2, out_features=3, bias=True)
  (1): ReLU()
) Sequential(
  (0): Linear(in_features=3, out_features=4, bias=True)
  (1): ReLU()
)
Sequential(
  (0): Sequential(
    (0): Linear(in_features=2, out_features=3, bias=True)
    (1): ReLU()
  )
  (1): Sequential(
    (0): Linear(in_features=3, out_features=4, bias=True)
    (1): ReLU()
  )
)
tensor([[0.0000, 0.4948, 0.0000, 0.5853],
        [2.3093, 3.0341, 2.2673, 3.1245]])


### ModuleList VS Sequential

In [25]:
fc=nn.Linear(3,3)
layer_list = [fc for _ in range(5)]
layers1 = nn.Sequential(*layer_list)
layers2 = nn.ModuleList(layer_list)
print(layers1)
print()
print(layers2)

x=torch.randn(1,3)
print(layers1(x))
print()

# print(layers2(x)) # error!
for layer in layers2:
    x = layer(x)
print(x)

Sequential(
  (0): Linear(in_features=3, out_features=3, bias=True)
  (1): Linear(in_features=3, out_features=3, bias=True)
  (2): Linear(in_features=3, out_features=3, bias=True)
  (3): Linear(in_features=3, out_features=3, bias=True)
  (4): Linear(in_features=3, out_features=3, bias=True)
)

ModuleList(
  (0-4): 5 x Linear(in_features=3, out_features=3, bias=True)
)
tensor([[ 0.5331, -0.0616, -0.7057]], grad_fn=<AddmmBackward0>)

tensor([[ 0.5331, -0.0616, -0.7057]], grad_fn=<AddmmBackward0>)


In [32]:
# 그냥 리스트 사용
class testNet(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.Module_List = [nn.Linear(3,3), nn.Linear(3,3)]
        
    def forward(self, x):
        for layer in self.Module_List:
            x = layer(x)
        return x
    
    
model = testNet()

print(model(torch.randn(1,3)))

print(model) # 그냥 리스트로 하면 등록이 안돼있다!!

#optimizer = optim.Adam(model.parameters(), lr = 0.1) # 등록이 안돼있으면 parameter를 못 찾는다..!

print(list(model.parameters())) # 비워있음..

tensor([[ 0.3533, -0.4838,  0.7325]], grad_fn=<AddmmBackward0>)
testNet()
[]


In [34]:
# module list 사용
class testNet(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.Module_List = nn.ModuleList([nn.Linear(3,3), nn.Linear(3,3)])
        
    def forward(self, x):
        for layer in self.Module_List:
            x = layer(x)
        return x

In [35]:
model = testNet()

print(model(torch.randn(1,3)))

print(model) 

#optimizer = optim.Adam(model.parameters(), lr = 0.1)

print(list(model.parameters())) 

tensor([[ 0.4257, -0.5140,  0.6777]], grad_fn=<AddmmBackward0>)
testNet(
  (Module_List): ModuleList(
    (0-1): 2 x Linear(in_features=3, out_features=3, bias=True)
  )
)
[Parameter containing:
tensor([[-0.3486, -0.3962,  0.0022],
        [-0.0808, -0.3762, -0.3150],
        [-0.5452, -0.4015, -0.1909]], requires_grad=True), Parameter containing:
tensor([-0.5507,  0.4061, -0.0096], requires_grad=True), Parameter containing:
tensor([[-0.2448, -0.4264, -0.4799],
        [ 0.1065,  0.4349, -0.1599],
        [-0.4951,  0.3328, -0.5651]], requires_grad=True), Parameter containing:
tensor([-0.0367, -0.4300,  0.0891], requires_grad=True)]


In [36]:
# 그럼 nn.Sequential 쓰고 말지 왜 굳이 nn.ModuleList?
class small_block(nn.Module):
    def __init__(self):
        super().__init__()
        self.block_x = nn.Linear(1,1)
        self.block_y = nn.Linear(1,1)

    def forward(self, x, y):
        x = self.block_x(x)
        y = self.block_y(y)
        return x, y

block = small_block()
print(block)

small_block(
  (block_x): Linear(in_features=1, out_features=1, bias=True)
  (block_y): Linear(in_features=1, out_features=1, bias=True)
)


In [37]:
model = nn.Sequential(block, block)
print(model)

Sequential(
  (0): small_block(
    (block_x): Linear(in_features=1, out_features=1, bias=True)
    (block_y): Linear(in_features=1, out_features=1, bias=True)
  )
  (1): small_block(
    (block_x): Linear(in_features=1, out_features=1, bias=True)
    (block_y): Linear(in_features=1, out_features=1, bias=True)
  )
)


In [ ]:
# model(torch.randn(1), torch.randn(1)) # error!
# nn.Sequential 이 가지고 있는 forward 함수를 call 하기 때문에 입력을 두 개 넣으면 안된다!!

In [ ]:
model = nn.ModuleList([block,block])
x = torch.randn(1)
y = torch.randn(1)
for block in model: # forward 역할
    x, y = block(x,y)
print(x, y)